# Matrix factorization for collaborative filtering, from scratch (numpy only)

This notebook goes with `notes.md` §13.3-§13.4. It generates a synthetic
ratings matrix with known low-rank structure, hides most entries, and fits
the biased matrix-factorization model
$\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$ on the observed entries only,
in two ways: SGD over observed ratings, and ALS, where every block update is a
ridge regression. Both are compared with simple baselines. A final sweep over
the rank $r$ shows test error is lowest at the true rank.

All randomness comes from fixed seeds, and the notebook runs in under a minute
on a CPU.

In [1]:
import numpy as np

np.random.seed(42)

## 1. Synthetic data: low rank plus biases plus noise, mostly missing

The full matrix is $200 \times 120$ (users by items) with
$R = \mu + b_u + b_i + P Q^\top + \text{noise}$: global mean 3.5, user and item
biases with standard deviation 0.5, a rank-5 interaction term, and noise with
standard deviation 0.4. Only about 25% of entries are observed (real data is
far sparser), and the observed entries are split 80/20 into train and test.
The noise level sets a floor: no model can get test RMSE much below 0.4.

In [2]:
M_USERS, N_ITEMS, TRUE_RANK = 200, 120, 5
NOISE_STD = 0.4
OBS_FRAC = 0.25

mu_true = 3.5
bu_true = 0.5 * np.random.randn(M_USERS)
bi_true = 0.5 * np.random.randn(N_ITEMS)
P_true = np.random.randn(M_USERS, TRUE_RANK) / np.sqrt(TRUE_RANK)
Q_true = np.random.randn(N_ITEMS, TRUE_RANK)
R_full = (mu_true + bu_true[:, None] + bi_true[None, :] + P_true @ Q_true.T
          + NOISE_STD * np.random.randn(M_USERS, N_ITEMS))

obs_mask = np.random.rand(M_USERS, N_ITEMS) < OBS_FRAC
obs_u, obs_i = np.where(obs_mask)
perm = np.random.permutation(len(obs_u))
n_train = int(0.8 * len(perm))
tr, te = perm[:n_train], perm[n_train:]
u_tr, i_tr, r_tr = obs_u[tr], obs_i[tr], R_full[obs_u[tr], obs_i[tr]]
u_te, i_te, r_te = obs_u[te], obs_i[te], R_full[obs_u[te], obs_i[te]]
train = (u_tr, i_tr, r_tr)
print(f"matrix {M_USERS}x{N_ITEMS} = {M_USERS * N_ITEMS} entries; "
      f"observed {len(obs_u)} ({len(obs_u) / (M_USERS * N_ITEMS):.1%}); "
      f"train {len(tr)}, test {len(te)}")

matrix 200x120 = 24000 entries; observed 6003 (25.0%); train 4802, test 1201


## 2. Baselines

Two models with no interaction term. The global mean predicts the same value
for every entry. The item mean is shrunk toward the global mean with 5
pseudo-ratings, $(\text{sum} + 5\bar r)/(\text{count} + 5)$, so an item with
few ratings does not get an extreme mean (the same shrinkage idea as ridge).
The item mean should beat the global mean, since item biases are part of the
data.

In [3]:
def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))


g_mean = r_tr.mean()
BETA = 5.0
item_sum = np.bincount(i_tr, weights=r_tr, minlength=N_ITEMS)
item_cnt = np.bincount(i_tr, minlength=N_ITEMS)
item_mean = (item_sum + BETA * g_mean) / (item_cnt + BETA)

baseline_rows = [
    ("global mean", rmse(np.full_like(r_tr, g_mean), r_tr),
     rmse(np.full_like(r_te, g_mean), r_te)),
    ("item mean (shrunk)", rmse(item_mean[i_tr], r_tr), rmse(item_mean[i_te], r_te)),
]
print(f"{'model':<22} {'train RMSE':>12} {'test RMSE':>12}")
for name, tr_r, te_r in baseline_rows:
    print(f"{name:<22} {tr_r:>12.4f} {te_r:>12.4f}")

model                    train RMSE    test RMSE
global mean                  1.2858       1.2370
item mean (shrunk)           1.1721       1.1275


## 3. MF with biases, trained by SGD (notes §13.4)

For each observed rating in random order, compute the error
$e_{ui} = r_{ui} - \hat r_{ui}$ and take one gradient step on the regularized
squared error: $b_u \mathrel{+}= \eta(e_{ui} - \lambda b_u)$, the same for
$b_i$, $p_u \mathrel{+}= \eta(e_{ui} q_i - \lambda p_u)$ and
$q_i \mathrel{+}= \eta(e_{ui} p_u - \lambda q_i)$ (using the old $p_u$). This
runs 100 epochs at rank 5; with only 40 epochs SGD has not converged yet
(test RMSE is then about 0.65). Expect test RMSE well below both baselines and
above the 0.4 noise floor.

In [4]:
def mf_sgd(train, rank, n_epochs=100, lr=0.02, lam=0.05, seed=0):
    u_tr, i_tr, r_tr = train
    rng = np.random.default_rng(seed)
    mu = r_tr.mean()
    bu = np.zeros(M_USERS)
    bi = np.zeros(N_ITEMS)
    P = 0.1 * rng.standard_normal((M_USERS, rank))
    Q = 0.1 * rng.standard_normal((N_ITEMS, rank))
    idx = np.arange(len(r_tr))
    for _ in range(n_epochs):
        rng.shuffle(idx)
        for t in idx:
            u, i, r = u_tr[t], i_tr[t], r_tr[t]
            e = r - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += lr * (e - lam * bu[u])
            bi[i] += lr * (e - lam * bi[i])
            pu = P[u].copy()
            P[u] += lr * (e * Q[i] - lam * P[u])
            Q[i] += lr * (e * pu - lam * Q[i])
    return mu, bu, bi, P, Q


def predict(model, u, i):
    mu, bu, bi, P, Q = model
    return mu + bu[u] + bi[i] + np.einsum("tk,tk->t", P[u], Q[i])


sgd_model = mf_sgd(train, rank=TRUE_RANK)
sgd_row = ("MF + biases (SGD)", rmse(predict(sgd_model, u_tr, i_tr), r_tr),
           rmse(predict(sgd_model, u_te, i_te), r_te))
print(f"SGD: train RMSE = {sgd_row[1]:.4f}, test RMSE = {sgd_row[2]:.4f}")

SGD: train RMSE = 0.3359, test RMSE = 0.5238


## 4. MF with biases, trained by ALS (notes §13.4)

ALS updates one block at a time, and each update is the exact minimizer of
the objective over that block with the others fixed. A user bias is a 1-D
ridge problem: $b_u = \sum(\text{residuals}) / (n_u + \lambda_b)$. A user's
factor vector is a ridge regression on the factor vectors of the items that
user rated, $p_u = (Q_u^\top Q_u + \lambda I)^{-1} Q_u^\top r_u$, where $r_u$
holds that user's ratings minus $\mu + b_u + b_i$. Items are updated the same
way with the roles swapped. This runs 15 sweeps at rank 5 and should reach
about the same test RMSE as SGD.

In [5]:
def mf_als(train, rank, n_sweeps=15, lam=1.0, lam_b=5.0, seed=0):
    u_tr, i_tr, r_tr = train
    rng = np.random.default_rng(seed)
    mu = r_tr.mean()
    bu = np.zeros(M_USERS)
    bi = np.zeros(N_ITEMS)
    P = 0.1 * rng.standard_normal((M_USERS, rank))
    Q = 0.1 * rng.standard_normal((N_ITEMS, rank))

    items_of = [np.where(u_tr == u)[0] for u in range(M_USERS)]  # each user's ratings
    users_of = [np.where(i_tr == i)[0] for i in range(N_ITEMS)]  # each item's ratings
    I = np.eye(rank)

    for _ in range(n_sweeps):
        # user biases: exact 1-D ridge solution
        interact = np.einsum("tk,tk->t", P[u_tr], Q[i_tr])
        resid = r_tr - (mu + bi[i_tr] + interact)
        bu = np.bincount(u_tr, weights=resid, minlength=M_USERS) / (
            np.bincount(u_tr, minlength=M_USERS) + lam_b)
        # item biases
        resid = r_tr - (mu + bu[u_tr] + interact)
        bi = np.bincount(i_tr, weights=resid, minlength=N_ITEMS) / (
            np.bincount(i_tr, minlength=N_ITEMS) + lam_b)
        # user factors: one ridge regression per user
        for u in range(M_USERS):
            t = items_of[u]
            if len(t) == 0:
                continue
            Qu = Q[i_tr[t]]                                  # (n_u, rank)
            ru = r_tr[t] - (mu + bu[u] + bi[i_tr[t]])
            P[u] = np.linalg.solve(Qu.T @ Qu + lam * I, Qu.T @ ru)
        # item factors: one ridge regression per item
        for i in range(N_ITEMS):
            t = users_of[i]
            if len(t) == 0:
                continue
            Pu = P[u_tr[t]]
            ri = r_tr[t] - (mu + bu[u_tr[t]] + bi[i])
            Q[i] = np.linalg.solve(Pu.T @ Pu + lam * I, Pu.T @ ri)
    return mu, bu, bi, P, Q


als_model = mf_als(train, rank=TRUE_RANK)
als_row = ("MF + biases (ALS)", rmse(predict(als_model, u_tr, i_tr), r_tr),
           rmse(predict(als_model, u_te, i_te), r_te))
print(f"ALS: train RMSE = {als_row[1]:.4f}, test RMSE = {als_row[2]:.4f}")

ALS: train RMSE = 0.3379, test RMSE = 0.5360


## 5. Comparison at the true rank

The table collects all four models. The assert checks that both MF variants
beat the better of the two baselines on test RMSE.

In [6]:
rows = baseline_rows + [sgd_row, als_row]
print(f"{'model':<22} {'train RMSE':>12} {'test RMSE':>12}")
for name, tr_r, te_r in rows:
    print(f"{name:<22} {tr_r:>12.4f} {te_r:>12.4f}")
print(f"(noise std = {NOISE_STD}: no model can get test RMSE much below {NOISE_STD})")

best_baseline = min(r[2] for r in baseline_rows)
for name, _, te_r in rows[2:]:
    assert te_r < best_baseline, f"{name} failed to beat the best baseline"
print("PASS: both MF variants beat both baselines on test RMSE.")

model                    train RMSE    test RMSE
global mean                  1.2858       1.2370
item mean (shrunk)           1.1721       1.1275
MF + biases (SGD)            0.3359       0.5238
MF + biases (ALS)            0.3379       0.5360
(noise std = 0.4: no model can get test RMSE much below 0.4)
PASS: both MF variants beat both baselines on test RMSE.


## 6. Rank sweep: test RMSE is lowest at the true rank

ALS is rerun with rank $r \in \{1, 2, 5, 10, 20\}$. Train RMSE can only go
down as $r$ grows, because a larger model has more parameters to fit the
training ratings. Test RMSE should be U-shaped: too high below rank 5 because
the model cannot represent the interactions, and rising again above rank 5
because the extra factors fit noise. The assert checks that the best test
RMSE is at $r = 5$.

In [7]:
print(f"{'rank':>6} {'train RMSE':>12} {'test RMSE':>12}")
sweep = {}
for r in [1, 2, 5, 10, 20]:
    model = mf_als(train, rank=r)
    tr_r = rmse(predict(model, u_tr, i_tr), r_tr)
    te_r = rmse(predict(model, u_te, i_te), r_te)
    sweep[r] = te_r
    marker = "  <- true rank" if r == TRUE_RANK else ""
    print(f"{r:>6} {tr_r:>12.4f} {te_r:>12.4f}{marker}")

best_rank = min(sweep, key=sweep.get)
print(f"best test RMSE at rank {best_rank} (true rank {TRUE_RANK})")
assert best_rank == TRUE_RANK, "rank sweep did not identify the true rank"
print("PASS: rank sweep identifies the true rank.")

  rank   train RMSE    test RMSE
     1       0.8716       0.9698
     2       0.7331       0.9053


     5       0.3379       0.5360  <- true rank


    10       0.2385       0.6455
    20       0.1464       0.6818
best test RMSE at rank 5 (true rank 5)
PASS: rank sweep identifies the true rank.
